# Devoir 1 - Pipeline NLP 

Ce notebook implemente un pipeline NLP complet pour transformer un texte juridique (Code de la route) en table structuree exploitable.

Objectif pratique:
- partir du texte brut extrait du PDF
- normaliser l'arabe pour rendre la recherche robuste
- segmenter les articles sur toutes les pages du document
- extraire des entites juridiques avec des patterns
- appliquer une etape ML (TF-IDF + clustering)
- proposer une approche pre-entrainee (spaCy, optionnelle)
- exporter un CSV final

Le traitement couvre automatiquement **toutes les pages** du PDF `code de la route MA52_05.pdf`.

## 1) Import des bibliotheques

Dans cette section:
- `PyPDF2` lit le document PDF
- `re` et `unicodedata` servent au nettoyage/normalisation
- `pandas` structure les donnees en DataFrame
- `scikit-learn` couvre la partie ML (vectorisation et clustering)

Idee importante: on separe clairement la partie extraction par regles (deterministe) et la partie ML (exploratoire).

In [1]:
from __future__ import annotations

import re
import unicodedata
import sys
from typing import Dict, List, Optional, Tuple

import pandas as pd
from IPython.display import display

# PDF backend: prefer PyPDF2, fallback to pdfplumber
PdfReader = None
_HAS_PYPDF2 = False
try:
    from PyPDF2 import PdfReader  # type: ignore
    _HAS_PYPDF2 = True
except Exception:
    try:
        import pdfplumber  # type: ignore
        _HAS_PYPDF2 = False
        print("Info: PyPDF2 not installed; pdfplumber will be used as fallback if available.")
    except Exception:
        _HAS_PYPDF2 = False
        print("Warning: Neither PyPDF2 nor pdfplumber are installed. Install PyPDF2 with `pip install PyPDF2` or run the provided setup script.")

%pip install scikit-learn

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

TASHKEEL_RE = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]')
BIDI_RE = re.compile(r'[\u200E\u200F\u202A-\u202E\u2066-\u2069]')
ZERO_WIDTH_RE = re.compile(r'[\u200B-\u200D\uFEFF]')
ARABIC_DIGITS_TRANS = str.maketrans('٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹', '01234567890123456789')

def normalize_text(text: str) -> str:
    text = unicodedata.normalize('NFKC', text or '')
    text = BIDI_RE.sub('', text)
    text = ZERO_WIDTH_RE.sub('', text)
    text = text.translate(ARABIC_DIGITS_TRANS)
    text = text.replace('ـ', '')
    text = text.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا').replace('ٱ', 'ا')
    text = text.replace('ؤ', 'و').replace('ئ', 'ي')
    text = text.replace('ة', 'ه').replace('ى', 'ي')
    text = TASHKEEL_RE.sub('', text)
    text = re.sub(r"[“”\"'`´«»]", "", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_line(line: str) -> str:
    line = normalize_text(line)
    line = re.sub(r'^[\-\–\—\•\·\*]+\s*', '', line)
    line = re.sub(r'^\d+\s*(?:الامانه العامة للحكومة|الامانه العامه للحكومة|الأمانة العامة للحكومة|الامانة العامة للحكومة)\b', '', line).strip()
    if re.fullmatch(r'(?:الامانه العامة للحكومة|الامانه العامه للحكومة|الأمانة العامة للحكومة|الامانة العامة للحكومة)', line):
        return ''
    if re.fullmatch(r'\d+', line):
        return ''
    return line

def read_pdf_lines(pdf_path: str) -> List[Dict[str, object]]:
    pages: List[Dict[str, object]] = []
    if _HAS_PYPDF2:
        reader = PdfReader(pdf_path)
        for page_number, page in enumerate(reader.pages, start=1):
            raw_text = page.extract_text() or ''
            cleaned_lines = [clean_line(line) for line in raw_text.splitlines()]
            cleaned_lines = [line for line in cleaned_lines if line]
            pages.append({
                'page': page_number,
                'raw_text': raw_text,
                'lines': cleaned_lines,
                'text': ' '.join(cleaned_lines),
            })
        return pages
    try:
        import pdfplumber  # type: ignore
    except Exception:
        raise RuntimeError("No PDF backend available. Install PyPDF2 (`pip install PyPDF2`) or pdfplumber (`pip install pdfplumber`).")
    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            raw_text = page.extract_text() or ''
            cleaned_lines = [clean_line(line) for line in raw_text.splitlines()]
            cleaned_lines = [line for line in cleaned_lines if line]
            pages.append({
                'page': page_number,
                'raw_text': raw_text,
                'lines': cleaned_lines,
                'text': ' '.join(cleaned_lines),
            })
    return pages

ARTICLE_LABEL_RE = r'(?:الماده|المادة|املاده|الاماده|ماده|مادة)'
ARTICLE_RE = re.compile(rf'(?:(?P<num_pre>\d{{1,4}})\s+{ARTICLE_LABEL_RE}|{ARTICLE_LABEL_RE}\s+(?P<num_post>\d{{1,4}}))')
SANCTION_TRIGGER_RE = re.compile(r'(يعاقب|غرامه|غرامة|خصم|سحب\s+نقط|عقوبه|عقوبة|حبس|سجن|مصادرة|توقيف|جزاء)', re.IGNORECASE)
AMENDE_CUES_RE = re.compile(r'(?:غرامه|غرامة|بغرامه|بغرامة|مبلغ\s+الغرامه|مبلغ\s+الغرامة|amende|fine)', re.IGNORECASE)
AMENDE_WORD_VALUES = [
    (re.compile(r'خمسة\s+وثلاثين\s+ألف|خمسة\s+وثلاثين\s+الف|35\.?000'), 35000),
    (re.compile(r'عشرين\s+ألف|عشرين\s+الف|20\.?000'), 20000),
    (re.compile(r'خمسة\s+آلاف|خمسة\s+الاف|5\.?000'), 5000),
    (re.compile(r'أربعة\s+آلاف|اربعة\s+الاف|4\.?000'), 4000),
    (re.compile(r'ثلاثة\s+آلاف|ثلاثة\s+الاف|3\.?000'), 3000),
    (re.compile(r'ألفين|الفين|ألفان|الفان|2\.?000'), 2000),
    (re.compile(r'ألف|الف|1\.?000'), 1000),
    (re.compile(r'خمسمائة|خمسمائه|خمسمئه|500'), 500),
    (re.compile(r'أربعمائة|اربعمائة|400'), 400),
    (re.compile(r'ثلاثمائة|300'), 300),
    (re.compile(r'مائتين|مائتا|مائتان|مائة|مئه|200'), 200),
]

def split_infraction_sanction(text: str) -> Tuple[str, str]:
    match = SANCTION_TRIGGER_RE.search(text)
    if not match:
        return text.strip(), ''
    return text[:match.start()].strip(), text[match.start():].strip()

def extract_article_number(line: str) -> Optional[str]:
    match = ARTICLE_RE.search(line)
    if not match:
        return None
    return match.group('num_pre') or match.group('num_post')

def strip_article_heading(line: str) -> str:
    return ARTICLE_RE.sub('', line, count=1).strip(' :-–—;،\t')

def extract_fine_value(text: str) -> Optional[float]:
    normalized = normalize_text(text)
    for cue in AMENDE_CUES_RE.finditer(normalized):
        window = normalized[cue.start(): cue.start() + 220]
        digit_match = re.search(r'(\d{1,6}(?:[.,]\d{1,2})?)\s*(?:درهم|دراهم|dh|dirhams?)?', window, re.IGNORECASE)
        if digit_match:
            value = digit_match.group(1).replace(',', '.')
            try:
                parsed = float(value)
                return int(parsed) if parsed.is_integer() else parsed
            except ValueError:
                pass
        for pattern, value in AMENDE_WORD_VALUES:
            if pattern.search(window):
                return value
    return None

def extract_fine_mentions(text: str, page_num: Optional[int] = None) -> List[Dict[str, object]]:
    normalized = normalize_text(text)
    hits: List[Dict[str, object]] = []
    for match in AMENDE_CUES_RE.finditer(normalized):
        window = normalized[max(0, match.start() - 40): min(len(normalized), match.start() + 240)]
        hits.append({
            'page': page_num,
            'cue': match.group(0),
            'amount': extract_fine_value(window),
            'snippet': window,
        })
    return hits

def segment_articles_from_page(text: str, page_num: int) -> List[Dict[str, object]]:
    lines = [clean_line(line) for line in text.splitlines()]
    lines = [line for line in lines if line]
    article_starts = [idx for idx, line in enumerate(lines) if ARTICLE_RE.search(line)]
    if not article_starts:
        return [{'article_id': f'page_{page_num}', 'article_text': text, 'infraction_desc': text, 'sanction_desc': '', 'start_page': page_num, 'end_page': page_num}]
    articles: List[Dict[str, object]] = []
    for i, start_idx in enumerate(article_starts):
        next_idx = article_starts[i + 1] if i + 1 < len(article_starts) else len(lines)
        block_lines = lines[start_idx:next_idx]
        block_text = ' '.join(block_lines).strip()
        article_id = extract_article_number(block_lines[0]) or f'{page_num}_{i + 1}'
        body = strip_article_heading(block_text)
        infraction_desc, sanction_desc = split_infraction_sanction(body)
        articles.append({'article_id': article_id, 'article_text': block_text, 'infraction_desc': infraction_desc, 'sanction_desc': sanction_desc, 'start_page': page_num, 'end_page': page_num})
    return articles

VEHICLE_PATTERNS = {
    'poids_lourd': re.compile(r'(شاحنه|شاحنة|نقل\s+البضايع|نقل\s+البضائع|وزن\s+ثقيل)'),
    'leger': re.compile(r'(سياره|سيارة|مركبه\s+خفيفه|مركبة\s+خفيفة|مركبات\s+خفيفة)'),
    'moto': re.compile(r'(دراجه\s+ناريه|دراجة\s+نارية|دراجه|دراجة)'),
    'bus': re.compile(r'(حافله|حافلة|نقل\s+عمومي|مركبة\s+نقل\s+الركاب)'),
    'farming': re.compile(r'(مركبه\s+فلاحيه|مركبة\s+فلاحية|غابويه|غابوية|أريبة|ارِيبة|عربة)'),
    'general_vehicle': re.compile(r'(مركبه\s+ذات\s+محرك|مركبة\s+ذات\s+محرك|مجموعه\s+مركبات|مجموعة\s+مركبات)'),
}

LOCATION_PATTERNS = {
    'route_publique': re.compile(r'(الطريق\s+العموميه|الطريق\s+العمومية|طريق\s+عموميه|طريق\s+عمومية|الطريق\s+العام)'),
    'territoire_national': re.compile(r'(التراب\s+الوطني|داخل\s+التراب\s+الوطني)'),
    'international': re.compile(r'(السير\s+الدولي|رخصه\s+دوليه|رخصة\s+دولية)'),
    'urbain': re.compile(r'(داخل\s+المدينه|داخل\s+المدينة|خارج\s+المدينه|خارج\s+المدينة)'),
    'autoroute': re.compile(r'(الطريق\s+السياره|الطريق\s+السيارة|الطريق\s+السريعة|الطريق\s+السريعه)'),
}

INFRACTION_PATTERNS = {
    'vitesse': re.compile(r'(سرعه|سرعة|تجاوز\s+السرعه|تجاوز\s+السرعة)'),
    'stationnement': re.compile(r'(وقوف|توقف|محظور\s+الوقوف|ممنوع\s+الوقوف)'),
    'permis': re.compile(r'(رخصه\s+السياقه|رخصة\s+السياقة|بدون\s+رخصه|بدون\s+رخصة|لا\s+يجوز)'),
}

KEYWORDS_LEXICON = ['سرعه','سرعة','وقوف','رخصه','رخصة','سياقه','سياقة','طريق','عموميه','عمومية']
OBLIGATION_CUES = ['يجب', 'يلزم', 'يتعين', 'لا يجوز', 'يمنع', 'لا يحق', 'ينبغي']
EXCEPTION_CUES = ['استثناء', 'باستثناء', 'إلا', 'الا', 'غير أنه', 'غير ان']
DEFINITION_CUES = ['يقصد', 'يعني', 'المقصود', 'تعتبر', 'تعني']

def parse_number(value: str):
    if value is None:
        return None
    value = value.replace(',', '.')
    try:
        if '.' in value:
            return float(value)
        return int(value)
    except ValueError:
        return None

def choose_label(patterns: Dict[str, re.Pattern], text: str, default: str) -> str:
    for label, pattern in patterns.items():
        if pattern.search(text):
            return label
    return default

def collect_keywords(text: str) -> List[str]:
    keywords: List[str] = []
    seen = set()
    for token in KEYWORDS_LEXICON:
        if token in text and token not in seen:
            seen.add(token)
            keywords.append(token)
    for label, pattern in INFRACTION_PATTERNS.items():
        if pattern.search(text) and label not in seen:
            seen.add(label)
            keywords.append(label)
    return keywords

def classify_role(text: str, sanction_desc: str) -> str:
    txt = f'{text} {sanction_desc}'.strip()
    if sanction_desc or re.search(r'يعاقب|غرامه|خصم|سحب', txt):
        return 'sanction'
    if any(cue in txt for cue in OBLIGATION_CUES):
        return 'obligation'
    if any(cue in txt for cue in EXCEPTION_CUES):
        return 'exception'
    if any(cue in txt for cue in DEFINITION_CUES):
        return 'definition'
    return 'information'

def classify_severity(text: str, sanction_desc: str, amende_fixe, points_retrait) -> str:
    score = 0
    if amende_fixe is not None:
        score += 1
        if amende_fixe >= 500:
            score += 1
        if amende_fixe >= 1000:
            score += 1
    if points_retrait is not None:
        score += 1
        if points_retrait >= 3:
            score += 1
    if sanction_desc:
        score += 1
    if re.search(r'(حبس|سجن|اعتقال)', text):
        score += 3
    if score >= 5:
        return 'high'
    if score >= 3:
        return 'medium'
    if score >= 1:
        return 'low'
    return 'information'

def extract_entities(article: Dict[str, object]) -> Dict[str, object]:
    text = str(article.get('article_text',''))
    infraction_desc = str(article.get('infraction_desc',''))
    sanction_desc = str(article.get('sanction_desc',''))
    amende = extract_fine_value(text)
    points = None
    role = classify_role(infraction_desc, sanction_desc)
    return {
        'article_id': article.get('article_id'),
        'infraction_desc': infraction_desc,
        'categorie_vehicule': choose_label(VEHICLE_PATTERNS, text, 'non_precise'),
        'amende_fixe': amende,
        'points_retrait': points,
        'mots_cles': collect_keywords(text),
        'role': role,
        'type_infraction': choose_label(INFRACTION_PATTERNS, text, 'general'),
        'niveau_gravite': classify_severity(text, sanction_desc, amende, points),
        'localisation': choose_label(LOCATION_PATTERNS, text, 'non_precise'),
        'sanction_desc': sanction_desc,
        'sanction_presente': bool(sanction_desc),
        'source_page': int(article.get('start_page', 0)),
        'source_page_end': int(article.get('end_page', 0)),
        'page_count': int(article.get('end_page', 0)) - int(article.get('start_page', 0)) + 1,
    }

def load_spacy_model():
    try:
        import spacy  # type: ignore
    except Exception:
        return None
    try:
        return spacy.load('xx_ent_wiki_sm')
    except Exception:
        return None

def extract_spacy_entities(text: str, nlp) -> List[str]:
    if nlp is None or not text:
        return []
    try:
        doc = nlp(text)
    except Exception:
        return []
    return [f'{ent.text}:{ent.label_}' for ent in doc.ents]

# Useful runtime info
print(sys.executable)

  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl (20.3 MB)
Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl (20.3 MB)
Using cached t

## 2) Chargement de toutes les pages du dataset

Ce bloc lit automatiquement l'ensemble du PDF pour ne pas limiter l'analyse aux quatre premieres pages.

Ce que fait le code:
- ouvre le PDF source
- lit toutes les pages disponibles
- stocke le texte brut de chaque page dans un dictionnaire `{page: texte}`

La sortie affiche la longueur de chaque page extraite pour verifier que la lecture s'est bien faite.

In [3]:
PDF_PATH = 'code de la route MA52_05.pdf'
pages_data = read_pdf_lines(PDF_PATH)
TOTAL_PAGES = len(pages_data)
raw_pages = {page['page']: page['raw_text'] for page in pages_data}
PAGES_TO_USE = list(raw_pages.keys())

print('Total pages:', TOTAL_PAGES)
print({p: len(t) for p, t in list(raw_pages.items())[:10]})
print('Last page length:', len(raw_pages[TOTAL_PAGES]))

Total pages: 126
{1: 210, 2: 1030, 3: 1944, 4: 1080, 5: 1757, 6: 1623, 7: 1673, 8: 1836, 9: 1956, 10: 2154}
Last page length: 1556


In [4]:
loaded_pages = sorted(raw_pages.keys())
print('Pages chargees:', loaded_pages[:5], '...', loaded_pages[-5:])
print('Nombre total de pages chargees:', len(loaded_pages))

for p in [1, 2, TOTAL_PAGES]:
    print(f'\n===== PAGE {p} (extrait) =====')
    print(raw_pages[p][:1200])

Pages chargees: [1, 2, 3, 4, 5] ... [122, 123, 124, 125, 126]
Nombre total de pages chargees: 126

===== PAGE 1 (extrait) =====
قلعتملا 52.05 مقر نوناقلا
، قرطلا ىلع ريسلا ةنودمب
ه ميمتتو هرييغت عقو امك
خ يراتب ةدطوم ةغيص
2024 ويلوي10
ليهست لجأ نم ةخسنلا هذه دادعإ مت
صوصنلاب لاإ جتحي لاو ،صنلا ةيئورقم
ةيمسرلا ةديرجلاب ةروشنلما اهتغيص يف

===== PAGE 2 (extrait) =====
ةموكحلل ةماعلا ةناملأا
52.05 مقر نوناقلا
1.10.07 مقر فيرشلا ريهظلا هذيفنتب رداصلا قرطلا ىلع ريسلا ةنودمب قلعتلما
،)2010 رياربف 11( 1431 رفص نم 26 خيراتب
هميمتتو هرييغت عقو امك
)2168 : ص ،)2010 سرام 25(1431 رخلآا عيبر 8 خيراتب 5824 ددع ر.ج(
لولأا باتكلا
ةيمومعلا قيرطلا ىلع ريسلا طورش
لولأا مسقلا
ةقايسلا ةصخر
لولأا بابلا
ةقايسلا ةصخر ةيمازلإ
1 ةدالما
ة يمومعلا قيرطلا ىلع تابكرم ةعومجم وأ كرحم تاذ ةبكرم قوسي نأ صخش يلأ زوجي لا
فنص بسانت ،ةرادلإا لبق نم ةملسمو ةيحلاصلا ةيراس ةقايسلل ةصخر ىلع لاصاح نكي مل ام
.اهقوسي يتلا تابكرلما ةعومجم وأ ةبكرلما
2 ةدالما
: هلاعأ ىلولأا ةدالما ماكحأ نم ءانثتسا
اهاصقأ ةدم للاخ ،ينطولا بارتلا لخا

## 3) Pretraitement arabe (Normalisation)

Pourquoi normaliser?
- les memes mots peuvent apparaitre sous des variantes graphiques
- les diacritiques (tashkeel) ajoutent du bruit
- certains caracteres bidi/OCR perturbent les regex

Transformations appliquees:
1. normalisation Unicode (`NFKC`)
2. suppression des marqueurs bidirectionnels
3. conversion des chiffres arabes vers chiffres ASCII
4. unification des variantes de Hamza/Alif
5. conversion de Ta Marbuta et Alif Maqsura
6. suppression des voyelles
7. compactage des espaces

Resultat: un texte plus stable pour la segmentation et l'extraction d'entites.

In [5]:
clean_pages = {p: normalize_text(t) for p, t in raw_pages.items()}
for p in [1, 2, 3, 4, TOTAL_PAGES]:
    print(f'PAGE {p} cleaned chars =', len(clean_pages[p]))

PAGE 1 cleaned chars = 210
PAGE 2 cleaned chars = 1030
PAGE 3 cleaned chars = 1944
PAGE 4 cleaned chars = 1052
PAGE 126 cleaned chars = 1552


In [6]:
amende_mentions = []
for page in pages_data:
    amende_mentions.extend(extract_fine_mentions(page['raw_text'], page['page']))

amende_df = pd.DataFrame(amende_mentions)
print('amende mentions found:', len(amende_df))
if not amende_df.empty:
    display(amende_df[['page', 'amount', 'cue', 'snippet']].head(25))
else:
    print('No amende mentions detected.')

amende mentions found: 0
No amende mentions detected.


## 4) Segmentation en articles + isolation infraction/sanction

Objectif:
- detecter les frontieres d'articles (ex: `المادة 3` ou variantes OCR)
- separer le contenu descriptif de la partie sanction

Logique utilisee:
- `ARTICLE_RE` detecte les en-tetes d'article, meme quand le PDF inverse l'ordre numero/mot
- chaque bloc entre deux articles devient une unite de travail
- un trigger de sanction (`يعاقب`, `غرامه`, `خصم`, etc.) sert a couper le texte en:
  - `infraction_desc`
  - `sanction_desc`

Cette etape prepare une structure semi-tabulaire avant l'extraction fine.

In [7]:
all_articles = []
for p in PAGES_TO_USE:
    all_articles.extend(segment_articles_from_page(clean_pages[p], p))

print('articles detectes:', len(all_articles))
if all_articles:
    display(pd.DataFrame(all_articles)[['article_id', 'start_page', 'infraction_desc', 'sanction_desc']].head(10))
else:
    print('Aucun article detecte. Verifier les variantes du token article.')

articles detectes: 126


,article_id,start_page,infraction_desc,sanction_desc
0,page_1,1,قلعتملا 52.05 مقر نوناقلا ، قرطلا يلع ريسلا هن...,
1,page_2,2,هموكحلل هماعلا هناملاا 52.05 مقر نوناقلا 1.10....,
2,page_3,3,هموكحلل هماعلا هناملاا 3 هدالما اهيلا راشلما ه...,
3,page_4,4,هموكحلل هماعلا هناملاا 6 هدالما لاغشلال هبيرا ...,
4,page_5,5,هموكحلل هماعلا هناملاا تاذ هيمومعلا لاغشلاا تا...,
5,page_6,6,هموكحلل هماعلا هناملاا : )E)C(( )ج( ه فنص هنور...,
6,page_7,7,هموكحلل هماعلا هناملاا ثلاثلا بابلا هقايسلا هص...,
7,page_8,8,هموكحلل هماعلا هناملاا هذه تبثتو .هقايسلا هصخر...,
8,page_9,9,هموكحلل هماعلا هناملاا .هلاعا 7 هدالما نم هريخ...,
9,page_10,10,هموكحلل هماعلا هناملاا 17 هدالما خيراتب 1.16.1...,


## 5) Extraction d'entites (Regex / Rule-based)

Ici on transforme chaque article en colonnes metier.

Entites principales:
- `article_id`
- `amende_fixe`
- `points_retrait`
- `categorie_vehicule`
- `localisation`
- `type_infraction`
- `mots_cles`

Points importants:
- les regex numeriques detectent les montants/points quand ils existent
- des dictionnaires de patterns ajoutent des infos semantiques (vehicule, localisation, type)
- un score `niveau_gravite` est calcule par regles explicites
- `role` classe le paragraphe (`definition`, `obligation`, `sanction`)

Cette approche est interpretable et facile a ajuster selon le document.

In [8]:
# Les fonctions d'extraction sont chargees depuis `pipeline_all_pages.py`.
# On peut verifier un echantillon directement ici.
if all_articles:
    sample = extract_entities(all_articles[0])
    print(sample)

{'article_id': 'page_1', 'infraction_desc': 'قلعتملا 52.05 مقر نوناقلا ، قرطلا يلع ريسلا هنودمب ه ميمتتو هرييغت عقو امك خ يراتب هدطوم هغيص 2024 ويلوي10 ليهست لجا نم هخسنلا هذه دادعا مت صوصنلاب لاا جتحي لاو ،صنلا هييورقم هيمسرلا هديرجلاب هروشنلما اهتغيص يف', 'categorie_vehicule': 'non_precise', 'amende_fixe': None, 'points_retrait': None, 'mots_cles': [], 'role': 'information', 'type_infraction': 'general', 'niveau_gravite': 'information', 'localisation': 'non_precise', 'sanction_desc': '', 'sanction_presente': False, 'source_page': 1, 'source_page_end': 1, 'page_count': 1}


## 6) Pipeline complet sur toutes les pages

Ce bloc applique la fonction d'extraction a tous les articles segmentes et construit le DataFrame final.

Ensuite, on impose l'ordre des colonnes pour obtenir un schema propre et coherent avec la consigne du devoir.

Verifiez ici:
- nombre de lignes (une ligne par article/regle detecte)
- presence des colonnes obligatoires
- qualite des champs textuels (`infraction_desc`, `sanction_desc`)

In [9]:
records = [extract_entities(article) for article in all_articles]
df = pd.DataFrame(records)

ordered_cols = [
    'article_id',
    'infraction_desc',
    'categorie_vehicule',
    'amende_fixe',
    'points_retrait',
    'mots_cles',
    'role',
    'type_infraction',
    'niveau_gravite',
    'localisation',
    'sanction_desc',
    'sanction_presente',
    'source_page',
    'source_page_end',
    'page_count',
]

if not df.empty:
    df = df[ordered_cols]

df

,article_id,infraction_desc,categorie_vehicule,amende_fixe,points_retrait,mots_cles,role,type_infraction,niveau_gravite,localisation,sanction_desc,sanction_presente,source_page,source_page_end,page_count
0,page_1,قلعتملا 52.05 مقر نوناقلا ، قرطلا يلع ريسلا هن...,non_precise,None,None,[],information,general,information,non_precise,,False,1,1,1
1,page_2,هموكحلل هماعلا هناملاا 52.05 مقر نوناقلا 1.10....,non_precise,None,None,[],information,general,information,non_precise,,False,2,2,1
2,page_3,هموكحلل هماعلا هناملاا 3 هدالما اهيلا راشلما ه...,non_precise,None,None,[],information,general,information,non_precise,,False,3,3,1
3,page_4,هموكحلل هماعلا هناملاا 6 هدالما لاغشلال هبيرا ...,non_precise,None,None,[],information,general,information,non_precise,,False,4,4,1
4,page_5,هموكحلل هماعلا هناملاا تاذ هيمومعلا لاغشلاا تا...,non_precise,None,None,[],information,general,information,non_precise,,False,5,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,page_122,هموكحلل هماعلا هناملاا سماخلا باتكلا هيماتخو ه...,non_precise,None,None,[],information,general,information,non_precise,,False,122,122,1
122,page_123,هموكحلل هماعلا هناملاا يناثلا مسقلا هيلاقتنا م...,non_precise,None,None,[],information,general,information,non_precise,,False,123,123,1
123,page_124,هموكحلل هماعلا هناملاا 312 هدالما رشن خيرات لب...,non_precise,None,None,[],information,general,information,non_precise,,False,124,124,1
124,page_125,هموكحلل هماعلا هناملاا 315 هدالما خيرات يف مهل...,non_precise,None,None,[],information,general,information,non_precise,,False,125,125,1


## 7) Optionnel - Vectorisation TF-IDF + Clustering

But de cette partie ML:
- representer chaque description d'infraction en vecteur numerique (`TF-IDF`)
- grouper automatiquement les articles similaires (`KMeans`)

Utilite pratique:
- explorer des familles d'articles sans etiquettes manuelles
- comparer la structure latente du corpus avec la classification par regles

Le nombre de clusters est limite dynamiquement pour eviter les erreurs sur petits echantillons.

In [10]:
if not df.empty:
    vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
    X = vectorizer.fit_transform(df['infraction_desc'].fillna('').astype(str))

    n_clusters = min(5, max(2, len(df) // 10 or 2))
    n_clusters = min(n_clusters, len(df))

    if n_clusters >= 2:
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df['cluster'] = kmeans.fit_predict(X)
        print('Shape TF-IDF:', X.shape)
        display(df[['article_id', 'type_infraction', 'cluster']].head(20))
    else:
        df['cluster'] = pd.NA
        print('Pas assez de lignes pour un clustering stable.')
else:
    print('Aucun article detecte.')

Shape TF-IDF: (126, 500)


,article_id,type_infraction,cluster
0,page_1,general,2
1,page_2,general,2
2,page_3,general,2
3,page_4,general,2
4,page_5,general,1
5,page_6,general,2
6,page_7,general,2
7,page_8,general,2
8,page_9,general,2
9,page_10,general,2


## 8) Optionnel - Approche modele pre-entraine spaCy

Cette section represente la 2eme approche demandee (modele pre-entraine).

Comportement:
- si spaCy et le modele sont installes: extraction d'entites nommees
- sinon: fallback propre sans interruption du notebook

Pourquoi utile:
- ajouter une couche statistique complementaire aux regex
- comparer `rule-based` vs `pretrained` dans votre analyse finale

In [11]:
if not df.empty:
    nlp_spacy = load_spacy_model()
    if nlp_spacy is None:
        df['spacy_entities'] = ''
        print('spaCy non disponible ou modele absent.')
    else:
        df['spacy_entities'] = df['infraction_desc'].fillna('').astype(str).apply(
            lambda text: extract_spacy_entities(text, nlp_spacy)
        )
        display(df[['article_id', 'spacy_entities']].head(20))

spaCy non disponible ou modele absent.


## 9) Export CSV final

Le DataFrame final est exporte en `UTF-8-SIG` pour bonne compatibilite (Excel/LibreOffice).

Le fichier `export_final.csv` est la sortie attendue du devoir: une ligne par regle/article detecte, avec toutes les colonnes extraites.

In [12]:
OUTPUT_CSV = 'export_final.csv'
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'Fichier genere: {OUTPUT_CSV} | lignes = {len(df)}')
print('Colonnes presentes:', list(df.columns))
print('Pages couvertes:', int(df['source_page'].min()), '->', int(df['source_page_end'].max()))

Fichier genere: export_final.csv | lignes = 126
Colonnes presentes: ['article_id', 'infraction_desc', 'categorie_vehicule', 'amende_fixe', 'points_retrait', 'mots_cles', 'role', 'type_infraction', 'niveau_gravite', 'localisation', 'sanction_desc', 'sanction_presente', 'source_page', 'source_page_end', 'page_count', 'cluster', 'spacy_entities']
Pages couvertes: 1 -> 126


## 10) Verification rapide

Checklist de validation:
1. toutes les pages du PDF sont bien lues
2. les articles sont detectes correctement
3. les colonnes obligatoires existent dans `df`
4. le CSV est genere sans erreur

Interpretation:
- certaines colonnes comme `amende_fixe` ou `points_retrait` peuvent rester vides si ces informations ne sont pas presentes
- le notebook reste executable cellule par cellule pour faciliter la correction et la revue par le professeur